# Notebook 06 — phase 3, verify the published `jv-mix-16k`

Run this in a **fresh session**, with **only the published dataset attached**, before any
GPU time is spent on notebook 06b. Accelerator: None. About five minutes.

## Why a separate notebook and not one more cell in the builder

Two notebooks *producing* the same artifact will drift, so phases 1 and 2 are one file
under a switch. A *verifier* is a consumer, and its independence from the producer is
exactly the property being tested — so it must be a separate file that shares no code
with the builder and re-derives everything from the mount.

This catches what an in-kernel check structurally cannot: a dataset published without
`_meta/`, an upload truncated by Kaggle (which has now bitten this project twice), a path
that shifts once Kaggle mounts it, a FLAC a different image will not read. Notebook 02's
verifier is the model; this one follows it.

## What it checks

1. **Inventory** — every shard header is read, not sampled: 16 kHz, mono, and long enough
   for the offsets that address it.
2. **Structure** — clips, items and shards are re-derived from the mount and must agree
   with each other; no clip used twice, no overlap, no item spanning two speakers or two
   shards. The `struct_hash` is recomputed and compared with the reviewed pass.
3. **The shipped digest describes the shipped data** — `dataset_findings.json` travelled
   with the artifact, so its counts are checked against the CSVs rather than trusted.
4. **Text** — the training targets are non-empty, digit-free, and each item's text really
   is its clips' text in order.
5. **Audio for real** — a sample of items is decoded by seeking into the shard exactly the
   way notebook 06b will, and must contain sound. A correct header can sit in front of
   digital silence.
6. **Experiment manifests** — every `mix_manifest_<E>.csv` references items that exist,
   and its hours match what the spec claims.
7. **Licence** — the notice a CC-BY-SA derivative is obliged to carry is present.

It ends with `DATASET VERIFIED` or `DO NOT USE THIS DATASET YET`. That line is the gate
for notebook 06b.


In [ ]:
# ============================================================
# CELL 1 - SETUP AND DISCOVERY (nothing is imported from the builder)
# ============================================================
import os, sys, csv, json, math, time, hashlib, collections, subprocess, shutil
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import soundfile as sf
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)

# The struct_hash printed by the pass that was REVIEWED. If the published data does not
# reproduce it, the review does not apply to what is on the mount.
EXPECT_STRUCT_HASH = "8640e1cfc11b794c35bdc06ce21390464568630fa761d3bbbc0886cb6302d888"

N_DECODE   = 60      # items decoded for real
SR_EXPECT  = 16000
NOTEBOOK_ID = "06-mix-verify"
OUT_DIR = Path("/kaggle/working") / "findings" / NOTEBOOK_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)

FINDINGS, CHECKS = {}, []
def rec(section, **kw): FINDINGS.setdefault(section, {}).update(kw)
def head(t): print("\n" + "=" * 74); print(t); print("=" * 74)
def check(name, ok, detail=""):
    CHECKS.append(dict(name=name, ok=bool(ok), detail=str(detail)[:200]))
    print("%-4s %-50s %s" % ("PASS" if ok else "FAIL", name, detail))

head("DISCOVERY")
INPUT = Path("/kaggle/input")
# Found by content, like everywhere else in this project: an items.csv with our header.
#
# There is more than one. The published output carries _kaggle-artifacts/<id>-lite.zip,
# which Kaggle extracts, so the metadata exists twice - and "_kaggle-artifacts" sorts
# before "shards", so taking the first hit would pick the copy that has no audio beside
# it and report all 91 shards missing. `jember-segments-16k` has the same shape and the
# builder was already bitten by it. So: score each candidate by the audio that actually
# sits under it, and take the one with the most.
cands = []
for q in sorted(INPUT.glob("**/items.csv")):
    try:
        hdr = q.open(encoding="utf-8", errors="replace").readline().lower()
    except Exception:
        continue
    if "item_id" in hdr and "shard" in hdr and "start_sample" in hdr:
        root = q.parent.parent.parent          # <root>/shards/_meta/items.csv
        n = len(list((root / "shards").glob("*.flac"))) if (root / "shards").is_dir() else 0
        cands.append((n, root, q.parent))
        print("  candidate: %-52s %d flac" % (str(root)[-52:], n))
ROOT, META, n = None, None, 0
if cands:
    n, ROOT, META = max(cands, key=lambda t: t[0])
if ROOT is None or n == 0:
    print("")
    print("!! No copy of jv-mix-16k has audio beside its metadata.")
    print("!! Attach the published dataset and re-run; if it IS attached, the publish")
    print("!! dropped the shards and the dataset has to be rebuilt.")
else:
    print("")
    print("dataset root :", ROOT)
    print("meta         :", META)
print("")
for d in sorted(p for p in (ROOT.iterdir() if ROOT else [])):
    k = len(list(d.iterdir())) if d.is_dir() else 0
    print("   %-28s %s" % (d.name, ("%d entries" % k) if d.is_dir() else "%d B" % d.stat().st_size))

CLIPS  = pd.read_csv(META / "clips.csv",  dtype={"clip_id": str, "item_id": str, "shard": str})
ITEMS  = pd.read_csv(META / "items.csv",  dtype={"item_id": str, "shard": str})
SHARDS = pd.read_csv(META / "shards.csv", dtype={"shard": str})
for df in (CLIPS, ITEMS):
    df["text_folded"] = df["text_folded"].fillna("").astype(str)
print("")
print("clips %d | items %d | shards %d" % (len(CLIPS), len(ITEMS), len(SHARDS)))
rec("env", root=str(ROOT), n_metadata_copies=len(cands), n_flac_beside_meta=n,
    n_clips=len(CLIPS), n_items=len(ITEMS), n_shards=len(SHARDS),
    python=sys.version.split()[0], soundfile=sf.__version__)


## 2 — Inventory: every shard header, not a sample

91 headers is cheap to read in full, and sampling is how a truncated upload survives a
check. Each shard must be 16 kHz mono and hold at least as many frames as the largest
offset that addresses it — a shard one sample short would fail only for the last item,
in the middle of a training run.


In [ ]:
# ============================================================
# CELL 2 - INVENTORY
# ============================================================
head("INVENTORY")
need = ITEMS.groupby("shard")["end_sample"].max().to_dict()
rows, missing, bad_fmt, short = [], [], [], []
t0 = time.time()
for s in sorted(set(ITEMS["shard"])):
    p = ROOT / "shards" / s
    if not p.exists():
        missing.append(s); continue
    try:
        info = sf.info(str(p))
    except Exception as e:
        bad_fmt.append((s, str(e)[:60])); continue
    if info.samplerate != SR_EXPECT or info.channels != 1:
        bad_fmt.append((s, "%d Hz, %d ch" % (info.samplerate, info.channels)))
    if info.frames < need.get(s, 0):
        short.append((s, info.frames, int(need.get(s, 0))))
    rows.append(dict(shard=s, frames=info.frames, dur_s=info.frames / float(SR_EXPECT),
                     bytes=p.stat().st_size, fmt=info.format, subtype=info.subtype))
INV = pd.DataFrame(rows)
print("read %d headers in %.1f s" % (len(INV), time.time() - t0))
if len(INV):
    print("total audio  : %.2f h across %.2f GB"
          % (INV["dur_s"].sum() / 3600, INV["bytes"].sum() / 1e9))
    print("formats      :", dict(collections.Counter(zip(INV["fmt"], INV["subtype"]))))

check("every shard named by the manifest exists", not missing, "%d missing" % len(missing))
check("every shard is 16 kHz mono FLAC", not bad_fmt, str(bad_fmt[:3]))
check("every shard is long enough for its offsets", not short, str(short[:3]))
check("no stray or extra shard files",
      len(list((ROOT / "shards").glob("*.flac"))) == len(set(ITEMS["shard"])),
      "%d on disk, %d in manifest"
      % (len(list((ROOT / "shards").glob("*.flac"))), len(set(ITEMS["shard"]))))
rec("inventory", n_shards=len(INV), hours=float(INV["dur_s"].sum() / 3600) if len(INV) else 0,
    gb=float(INV["bytes"].sum() / 1e9) if len(INV) else 0,
    n_missing=len(missing), n_bad_format=len(bad_fmt), n_short=len(short))


## 3 — Structure, re-derived rather than trusted

Every relation the builder claims is recomputed here from the three CSVs on the mount.
The `struct_hash` at the end is the same function of the same three columns the builder
used, so if the published data is the reviewed data, the two strings are identical.


In [ ]:
# ============================================================
# CELL 3 - STRUCTURE
# ============================================================
head("STRUCTURE")
check("no clip appears twice", CLIPS["clip_id"].is_unique,
      "%d duplicates" % int(CLIPS["clip_id"].duplicated().sum()))
check("no item id appears twice", ITEMS["item_id"].is_unique)
check("every clip belongs to a known item",
      set(CLIPS["item_id"]) <= set(ITEMS["item_id"]),
      "%d orphans" % len(set(CLIPS["item_id"]) - set(ITEMS["item_id"])))
check("every item owns at least one clip",
      set(ITEMS["item_id"]) <= set(CLIPS["item_id"]),
      "%d empty items" % len(set(ITEMS["item_id"]) - set(CLIPS["item_id"])))
check("no item spans two speakers or folders",
      int(CLIPS.groupby("item_id")["group_id"].nunique().max()) == 1)
check("no item spans two shards",
      int(CLIPS.groupby("item_id")["shard"].nunique().max()) == 1)
check("clip counts agree between the two tables",
      bool((CLIPS.groupby("item_id").size().sort_index()
            == ITEMS.set_index("item_id")["n_clip"].sort_index()).all()))

ov = 0
for s, g in CLIPS.sort_values(["shard", "start_sample"]).groupby("shard"):
    ov += int((g["start_sample"].values[1:] < g["end_sample"].values[:-1]).sum())
check("no two clips overlap inside a shard", ov == 0, "%d overlaps" % ov)

span = CLIPS.groupby("item_id").agg(lo=("start_sample", "min"), hi=("end_sample", "max"))
j = ITEMS.set_index("item_id").join(span)
check("every clip sits inside its item's span",
      bool((j["start_sample"] <= j["lo"]).all() and (j["end_sample"] >= j["hi"]).all()))
check("no item exceeds Whisper's 30 s window",
      float(ITEMS["dur_s"].max()) <= 30.0, "max %.2f s" % ITEMS["dur_s"].max())
check("item durations match their sample spans",
      bool((((ITEMS["end_sample"] - ITEMS["start_sample"]) / float(SR_EXPECT)
             - ITEMS["dur_s"]).abs() < 0.01).all()))
check("every path points inside shards/",
      bool(ITEMS["path"].astype(str).str.startswith("shards/").all()))

h = hashlib.sha256(("\n".join("%s|%s|%s" % (r.clip_id, r.item_id, r.shard)
                              for r in CLIPS.itertuples())).encode("utf-8")).hexdigest()
print("")
print("struct_hash recomputed: %s" % h)
print("struct_hash expected  : %s" % EXPECT_STRUCT_HASH)
check("structure reproduces the reviewed pass", h == EXPECT_STRUCT_HASH)
rec("structure", struct_hash=h, expected=EXPECT_STRUCT_HASH, overlaps=ov,
    max_item_s=float(ITEMS["dur_s"].max()))


## 4 — Does the shipped digest describe the shipped data?

`dataset_findings.json` travelled inside the dataset, which makes it a claim, not
evidence. Every count in it is checked against the CSVs that sit beside it.


In [ ]:
# ============================================================
# CELL 4 - DIGEST vs DATA
# ============================================================
head("DIGEST")
dj = ROOT / "dataset_findings.json"
if not dj.exists():
    check("dataset_findings.json shipped with the data", False, "not found")
    D = {}
else:
    D = json.load(open(dj, encoding="utf-8"))
    check("dataset_findings.json shipped with the data", True, "%d B" % dj.stat().st_size)
    a = D.get("audio", {})
    p = D.get("packing", {})
    print("digest says   : %s clips, %s items, %s shards, %s decode failures"
          % (a.get("n_clips"), a.get("n_items"), a.get("n_shards"), a.get("n_decode_failures")))
    print("mount holds   : %d clips, %d items, %d shards" % (len(CLIPS), len(ITEMS), len(SHARDS)))
    check("digest clip count matches the mount",  a.get("n_clips")  == len(CLIPS))
    check("digest item count matches the mount",  a.get("n_items")  == len(ITEMS))
    check("digest shard count matches the mount", a.get("n_shards") == len(SHARDS))
    check("the build recorded no decode failures", (a.get("n_decode_failures") or 0) == 0,
          "%s" % a.get("n_decode_failures"))
    check("digest was written by phase 2, not phase 1",
          bool(D.get("env", {}).get("write_audio")), "write_audio=%s"
          % D.get("env", {}).get("write_audio"))
    check("digest struct_hash agrees with the recomputed one",
          (a.get("struct_hash") or p.get("struct_hash")) == EXPECT_STRUCT_HASH)
rec("digest", present=dj.exists(), keys=sorted(D.keys()))


## 5 — Text

The training target must be non-empty and free of numerals — the official reference never
writes digits, and a mixture that reintroduces them undoes work notebook 08 has to do.
Item text is also re-derived from its clips, so a mismatch between the two tables cannot
hide.


In [ ]:
# ============================================================
# CELL 5 - TEXT
# ============================================================
head("TEXT")
check("no item has empty text", int((ITEMS["text_folded"].str.strip() == "").sum()) == 0)
check("no clip has empty text", int((CLIPS["text_folded"].str.strip() == "").sum()) == 0)
n_dig = int(ITEMS["text_folded"].str.contains(r"\d", regex=True).sum())
check("no digits in any training target", n_dig == 0, "%d items with a digit" % n_dig)

rebuilt = (CLIPS.sort_values(["item_id", "start_sample"])
                .groupby("item_id")["text_folded"]
                .apply(lambda s: " ".join(x.strip() for x in s).strip()))
cmp = ITEMS.set_index("item_id")["text_folded"].str.strip().sort_index()
bad = (rebuilt.sort_index() != cmp)
check("item text is exactly its clips' text in order", int(bad.sum()) == 0,
      "%d mismatched" % int(bad.sum()))
if int(bad.sum()):
    for k in bad[bad].index[:3]:
        print("   item", k); print("     items.csv:", cmp[k][:90]); print("     clips.csv:", rebuilt[k][:90])

wl = ITEMS["text_folded"].str.split().map(len)
print("words per item: median %d, max %d" % (int(wl.median()), int(wl.max())))
check("no item text is absurdly long for a 30 s window", int(wl.max()) < 200,
      "max %d words" % int(wl.max()))
rec("text", n_digit_items=n_dig, n_text_mismatch=int(bad.sum()),
    words_median=int(wl.median()), words_max=int(wl.max()))


## 6 — Decode for real, the way notebook 06b will

A correct header can sit in front of digital silence, and an offset table can be
internally consistent while pointing at the wrong samples. This reads a sample of items
by seeking into the shard — the exact call the training loader makes — and requires that
what comes back is the right length and is not silent.


In [ ]:
# ============================================================
# CELL 6 - REAL DECODE
# ============================================================
head("DECODE")
rs = np.random.RandomState(20260905)
# Sample across shards, not just the first ones, and always include the last item of each
# of a few shards - the end of a file is where truncation shows up.
idx = rs.choice(len(ITEMS), size=min(N_DECODE, len(ITEMS)), replace=False)
tail = ITEMS.sort_values("end_sample").groupby("shard").tail(1).head(15)
sample = pd.concat([ITEMS.iloc[idx], tail]).drop_duplicates("item_id")

silent, wrong_len, failed, rms = [], [], [], []
t0 = time.time()
for r in sample.to_dict("records"):
    try:
        y, sr = sf.read(str(ROOT / r["path"]), start=int(r["start_sample"]),
                        stop=int(r["end_sample"]), dtype="float32")
    except Exception as e:
        failed.append((r["item_id"], str(e)[:60])); continue
    want = int(r["end_sample"]) - int(r["start_sample"])
    if abs(len(y) - want) > 1:
        wrong_len.append((r["item_id"], len(y), want))
    a = float(np.sqrt(np.mean(np.square(y.astype(np.float64))))) if y.size else 0.0
    rms.append(a)
    if y.size == 0 or float(np.abs(y).max()) < 1e-4:
        silent.append(r["item_id"])
print("decoded %d items in %.1f s" % (len(sample) - len(failed), time.time() - t0))
if rms:
    print("rms: median %.4f  min %.5f  max %.4f"
          % (float(np.median(rms)), float(np.min(rms)), float(np.max(rms))))
check("every sampled item decodes", not failed, str(failed[:3]))
check("decoded length matches the offsets", not wrong_len, str(wrong_len[:3]))
check("no sampled item is silent", not silent, "%d silent: %s" % (len(silent), silent[:3]))

# Read one whole shard end to end: seeking can succeed on a file that cannot be streamed.
s0 = sorted(set(ITEMS["shard"]))[0]
y, sr = sf.read(str(ROOT / "shards" / s0), dtype="float32")
check("a whole shard reads end to end", sr == SR_EXPECT and y.ndim == 1 and y.size > 0,
      "%s: %d samples at %d Hz" % (s0, y.size, sr))
rec("decode", n_sampled=len(sample), n_failed=len(failed), n_silent=len(silent),
    n_wrong_length=len(wrong_len),
    rms_median=(float(np.median(rms)) if rms else None))


## 7 — Experiment manifests and the licence notice

The manifests are what notebook 06b actually reads, so a manifest naming an item that does
not exist would fail there rather than here. And a CC-BY-SA derivative distributed without
its attribution notice is a licence breach, not a cosmetic omission.


In [ ]:
# ============================================================
# CELL 7 - MANIFESTS AND LICENCE
# ============================================================
head("MANIFESTS")
mans = sorted(ROOT.glob("mix_manifest_*.csv"))
check("experiment manifests shipped", len(mans) >= 4, "%d found" % len(mans))
known = set(ITEMS["item_id"])
rows = []
for m in mans:
    eid = m.stem.replace("mix_manifest_", "")
    M = pd.read_csv(m, dtype={"item_id": str})
    unknown = set(M["item_id"]) - known
    spec_p = ROOT / ("mix_spec_%s.json" % eid)
    spec = json.load(open(spec_p, encoding="utf-8")) if spec_p.exists() else {}
    claimed = float(spec.get("add_hours") or 0.0)
    actual  = float(M["dur_s"].sum() / 3600.0)
    rows.append(dict(exp=eid, items=len(M), unknown=len(unknown), spec=spec_p.exists(),
                     claimed_h=claimed, actual_h=actual,
                     drift_pct=100.0 * (actual - claimed) / max(claimed, 1e-9)))
    check("%s references only existing items" % eid, not unknown,
          "%d unknown" % len(unknown))
    # The item COUNT is the exact contract - it is the selection notebook 06b acts on.
    check("%s item count matches its spec" % eid, int(spec.get("n_add_items", -1)) == len(M),
          "spec %s, manifest %d" % (spec.get("n_add_items"), len(M)))
    # Hours are NOT exact and are not meant to be. The spec is written before any audio is
    # decoded, from ffprobe container durations; the manifest is rewritten afterwards from
    # the decoded lengths. Opus reports ~60 ms less than it decodes, so the TTS arm drifts
    # by ~1 %. Reported, bounded, and never used for selection - 06b reads offsets from
    # items.csv, never hours from a spec.
    drift = abs(actual - claimed) / max(claimed, 1e-9)
    check("%s hours are within the pre-decode estimate" % eid, drift < 0.02,
          "spec %.2f h, manifest %.2f h (%+.1f %%)"
          % (claimed, actual, 100.0 * (actual - claimed) / max(claimed, 1e-9)))
if rows:
    print("")
    print(pd.DataFrame(rows).to_string(index=False))

head("LICENCE")
lic, rdm = ROOT / "LICENSE.txt", ROOT / "README.md"
check("LICENSE.txt ships with the data", lic.exists())
check("README.md ships with the data", rdm.exists())
if lic.exists():
    txt = lic.read_text(encoding="utf-8")
    check("the notice names the licence and both sources",
          ("CC-BY-SA-4.0" in txt) and ("mozilladatacollective.com" in txt),
          txt.splitlines()[0][:70])
rec("manifests", table=rows, license=lic.exists(), readme=rdm.exists())


## 8 — Verdict

`DATASET VERIFIED` is the gate for notebook 06b. Anything else means the GPU stays off.


In [ ]:
# ============================================================
# CELL 8 - VERDICT
# ============================================================
head("VERDICT")
n_fail = sum(1 for c in CHECKS if not c["ok"])
print("%d checks, %d passed, %d failed" % (len(CHECKS), len(CHECKS) - n_fail, n_fail))
for c in CHECKS:
    if not c["ok"]:
        print("   FAILED: %-46s %s" % (c["name"], c["detail"]))
VERDICT = (">>> DATASET VERIFIED. Notebook 06b may mount it."
           if n_fail == 0 else
           ">>> DO NOT USE THIS DATASET YET. %d checks failed." % n_fail)
print("")
print(VERDICT)
rec("verdict", verdict=VERDICT, n_checks=len(CHECKS), n_failed=n_fail, checks=CHECKS)

def jsonable(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float("%.6g" % float(o))
    if isinstance(o, (np.bool_,)):    return bool(o)
    if isinstance(o, float):          return float("%.6g" % o)
    if isinstance(o, dict):  return {str(k): jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v) for v in o]
    if isinstance(o, Path): return str(o)
    return o

FIN = jsonable(FINDINGS)
json.dump(FIN, open(OUT_DIR / "dataset_findings.json", "w"), indent=1)
print("")
print("=" * 74)
print("BEGIN CLAUDE-HANDOFF-JSON")
print(json.dumps(FIN, ensure_ascii=False, separators=(",", ":")))
print("END CLAUDE-HANDOFF-JSON")
print("=" * 74)
